# PneumoShift: Phase 1 Data Audit
## Beyond Accuracy: Evaluating Robustness & Explainability Across Chest X-Ray Datasets

This notebook audits the clinical chest X-ray cohorts (**NIH ChestX-ray14** and **Stanford CheXpert**) used in the study.

### Core Audit Objectives:
1. **Zero Patient Overlap Verification:** Ensure strict patient-level separation between `train`, `val`, `test`, and `target_cal` partitions.
2. **Harmonized Shared Label Prevalences:** Quantify positive and uncertain rates across all 8 canonical classes.
3. **Pneumonia Imbalance (Headline Task):** Characterize pneumonia's low clinical prevalence (~1-3%) to motivate **AUPRC** reporting.
4. **Demographic & Projection Confounder Analysis:** Quantify age, sex, and projection (AP vs PA) distributions across cohorts.

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pneumoshift.data.harmonize_labels import (
    SHARED_CLASSES,
    PATHOLOGY_CLASSES,
    HEADLINE_CLASS,
    harmonize_nih_labels,
    harmonize_chexpert_labels,
)
from pneumoshift.data.splits import (
    create_synthetic_dataset,
    verify_patient_leakage,
)
from pneumoshift.data.audit import run_data_audit, format_markdown_audit_report

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
print("Shared Canonical Label Set:", SHARED_CLASSES)
print("Headline Classification Finding:", HEADLINE_CLASS)

### 1. Load Split Partitions
If raw NIH or CheXpert datasets are not yet downloaded in the local environment, we automatically fall back to the synthetic smoke test cohort to demonstrate the complete auditing protocol.

In [ ]:
nih_splits_dir = Path("data/splits/nih")
chex_splits_dir = Path("data/splits/chexpert")
synth_splits_dir = Path("data/synthetic/splits")

splits = {}
images_dir = None
dataset_source = None

if (nih_splits_dir / "nih_train.csv").is_file():
    print("Loading NIH ChestX-ray14 splits...")
    splits = {s: pd.read_csv(nih_splits_dir / f"nih_{s}.csv") for s in ["train", "val", "test", "target_cal"]}
    images_dir = Path("data/nih_cxr/images")
    dataset_source = "NIH ChestX-ray14"
elif (chex_splits_dir / "chexpert_train.csv").is_file():
    print("Loading Stanford CheXpert splits...")
    splits = {s: pd.read_csv(chex_splits_dir / f"chexpert_{s}.csv") for s in ["train", "val", "test", "target_cal"]}
    images_dir = Path("data/chexpert/CheXpert-v1.0-small")
    dataset_source = "Stanford CheXpert"
else:
    print("No raw splits found on disk. Generating synthetic cohort for demonstration...")
    synth_paths = create_synthetic_dataset("data/synthetic", num_samples=40, image_size=64, seed=42)
    splits = {s: pd.read_csv(p) for s, p in synth_paths.items()}
    images_dir = Path("data/synthetic/images")
    dataset_source = "Synthetic Demonstration Cohort"

# Verify Patient-Level Isolation
verify_patient_leakage(splits, patient_col="patient_id")
for name, df in splits.items():
    print(f"{name.upper()}: {len(df):,} images, {df['patient_id'].nunique():,} unique patients")

### 2. Comprehensive Data Audit
Execute `run_data_audit` to extract prevalences, demographics, views, and pixel statistics.

In [ ]:
audit_results = run_data_audit(
    splits=splits,
    images_dir=images_dir,
    output_dir="results/tables/audit_notebook",
    generate_grid=True,
)

md_summary = format_markdown_audit_report(audit_results)
from IPython.display import Markdown, display
display(Markdown(md_summary))

### 3. Prevalence Comparison Across Classes
Visualizing class frequencies highlights severe class imbalance for Pneumonia (~1-3%), emphasizing the statistical necessity of Area Under the Precision-Recall Curve (AUPRC).

In [ ]:
train_df = splits["train"]
rates = []
for cls in SHARED_CLASSES:
    if cls in train_df.columns:
        r = (train_df[cls] == 1.0).mean() * 100.0
        rates.append({"Finding": cls, "Prevalence (%)": r, "Is_Headline": (cls == HEADLINE_CLASS)})

prev_df = pd.DataFrame(rates).sort_values("Prevalence (%)", ascending=False)

plt.figure(figsize=(9, 4.5))
colors = ["#d95f02" if x else "#1f77b4" for x in prev_df["Is_Headline"]]
sns.barplot(data=prev_df, x="Prevalence (%)", y="Finding", palette=colors)
plt.title(f"{dataset_source}: Class Prevalence in Training Partition\n(Headline task 'Pneumonia' highlighted in orange)", fontsize=11)
plt.xlabel("Positive Prevalence (%)")
plt.ylabel("")
plt.tight_layout()
plt.savefig("results/figures/audit_prevalence_bar.png", dpi=200)
plt.show()

### 4. Demographic & Projection Confounders
- **Age Distribution**: Mean, median, and IQR.
- **View Position (AP vs. PA)**: Major institutional confounder (CheXpert is mostly AP ICU bed-bound; NIH is mostly PA ambulatory).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Age Distribution
valid_ages = train_df["age"][train_df["age"] >= 0]
if len(valid_ages) > 0:
    sns.histplot(valid_ages, kde=True, ax=axes[0], color="#2b5c8f", bins=20)
    axes[0].set_title(f"Age Distribution (Median: {valid_ages.median():.0f} yrs)")
    axes[0].set_xlabel("Patient Age (Years)")
else:
    axes[0].text(0.5, 0.5, "Age data not available", ha="center")

# View Projections (AP vs PA)
if "view_position" in train_df.columns:
    view_counts = train_df["view_position"].value_counts()
    axes[1].pie(
        view_counts.values,
        labels=view_counts.index,
        autopct="%1.1f%%",
        startangle=140,
        colors=["#4c72b0", "#dd8452", "#55a868"]
    )
    axes[1].set_title("Radiographic View Projections (AP vs. PA)")

plt.tight_layout()
plt.savefig("results/figures/audit_demographics_and_views.png", dpi=200)
plt.show()

### 5. Radiograph Sample Gallery
Visual inspection of representative chest radiographs with metadata and headline pneumonia status.

In [ ]:
from pneumoshift.data.audit import generate_audit_sample_grid

fig = generate_audit_sample_grid(
    df=train_df,
    images_dir=images_dir,
    output_path="results/figures/audit_radiograph_gallery.png",
    num_samples=8,
)
if fig:
    plt.show()